In [1]:
# Clean reset
!pip uninstall -y transformers peft accelerate -q

# Install compatible versions
!pip install -q \
    transformers==4.41.2 \
    accelerate==0.30.1 \
    peft==0.11.1 \
    sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.9 MB/s eta 0:00:00:00:01


In [2]:
!pip install -q seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
import transformers, peft, accelerate

print(transformers.__version__)
print(peft.__version__)
print(accelerate.__version__)

4.41.2
0.11.1
0.30.1


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "training" in dirs:
        print("Found at:", root)
        break

Found at: /kaggle/input/datasets/ajaykumarprasad123/jd-bias-detector-version3


In [5]:
import os, shutil

SRC  = "/kaggle/input/datasets/ajaykumarprasad123/jd-bias-detector-version3"
DEST = "/kaggle/working/jd-bias-detector"

# Copy project to working directory (writable)
if os.path.exists(DEST):
    shutil.rmtree(DEST)

shutil.copytree(SRC, DEST)
os.chdir(DEST)

print("CWD:", os.getcwd())
print("Files:", os.listdir("."))

CWD: /kaggle/working/jd-bias-detector
Files: ['training', 'requirements.txt', 'data']


In [6]:
import os
os.chdir("/kaggle/working/jd-bias-detector")
print("CWD:", os.getcwd())
print("Files:", os.listdir("."))

CWD: /kaggle/working/jd-bias-detector
Files: ['training', 'requirements.txt', 'data']


In [7]:
import os
os.chdir("/kaggle/working/jd-bias-detector")

for f in ["data/annotated/train.jsonl",
          "data/annotated/val.jsonl",
          "data/annotated/test.jsonl",
          "data/annotated/train_augmented.jsonl"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑  Deleted {f}")
    else:
        print(f"   Not found: {f}")

print("\nDone. Ready to regenerate.")

   Not found: data/annotated/train.jsonl
   Not found: data/annotated/val.jsonl
   Not found: data/annotated/test.jsonl
   Not found: data/annotated/train_augmented.jsonl

Done. Ready to regenerate.


In [8]:
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# Safe directory change
if os.path.exists("/kaggle/working/jd-bias-detector"):
    os.chdir("/kaggle/working/jd-bias-detector")

# Install datasets if needed
try:
    import datasets
except ImportError:
    !pip install -q datasets

# Run data prep
!python -m training.data_prep \
    --output_dir data/annotated \
    --source both \
    --n_synthetic 5000

⬇  Loading HuggingFace job description data ...
   Trying DBD-research-group/job-descriptions-500k ...
   DBD-research-group/job-descriptions-500k failed: Dataset 'DBD-research-group/job-descriptions-500k' doesn't exist on the Hub or c
   Trying swimming/job_descriptions ...
   swimming/job_descriptions failed: Dataset 'swimming/job_descriptions' doesn't exist on the Hub or cannot be access
   Trying vikp/job_postings ...
   vikp/job_postings failed: Dataset 'vikp/job_postings' doesn't exist on the Hub or cannot be accessed.
   Trying jacob-hugging-face/job-descriptions ...
README.md: 100%|██████████████████████████████| 24.0/24.0 [00:00<00:00, 149kB/s]
training_data.csv: 3.77MB [00:00, 58.5MB/s]
Generating train split: 100%|███████| 853/853 [00:00<00:00, 11009.62 examples/s]
   Added 844 samples from jacob-hugging-face/job-descriptions
   Trying elricwan/job-description ...
   elricwan/job-description failed: Dataset 'elricwan/job-description' doesn't exist on the Hub or cannot be acc

In [9]:
import json
import random
from collections import Counter, defaultdict

DATA_DIR = "data/annotated"

def load_split(split):
    path = f"{DATA_DIR}/{split}.jsonl"
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train = load_split("train")
val   = load_split("val")
test  = load_split("test")

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")


# ================= SAMPLE PRINT =================
def print_samples(data, n=3):
    print("\n🔹 Sample Data:\n")
    for sample in random.sample(data, n):
        print("TEXT:", sample["text"][:150])
        print("TOKENS:", sample["tokens"][:15])
        print("LABELS:", sample["labels"][:15])
        print("-"*80)

print_samples(train)


# ================= LABEL DISTRIBUTION =================
def label_distribution(data):
    counter = Counter(
        label
        for sample in data
        for label in sample["labels"]
    )
    return counter

print("\n🔹 Label Distribution (Train):")
for k, v in label_distribution(train).items():
    print(f"{k}: {v}")


# ================= CATEGORY DISTRIBUTION =================
def category_distribution(counter):
    cat_counter = defaultdict(int)
    for lbl, count in counter.items():
        if lbl.startswith("B-"):
            cat = lbl.split("-")[1]
            cat_counter[cat] += count
    return cat_counter

cat_counts = category_distribution(label_distribution(train))

print("\n🔹 Category Distribution (Train):")
for k, v in cat_counts.items():
    print(f"{k}: {v}")


# ================= BIAS VS NEUTRAL =================
def bias_ratio(data):
    biased = sum(1 for s in data if any(l != "O" for l in s["labels"]))
    return biased, len(data) - biased

b, n = bias_ratio(train)
print(f"\n🔹 Bias vs Neutral (Train): {b} biased | {n} neutral")


# ================= CHECK REPETITION =================
def top_phrases(data, top_k=10):
    phrases = Counter()

    for s in data:
        text = s["text"].lower()
        for phrase in ["fast-paced", "young", "rockstar", "ninja"]:
            if phrase in text:
                phrases[phrase] += 1

    return phrases.most_common(top_k)

print("\n🔹 Common Bias Phrases:")
for p, c in top_phrases(train):
    print(f"{p}: {c}")

Train: 2614 | Val: 322 | Test: 338

🔹 Sample Data:

TEXT: Our team is looking for a DevOps Engineer. You will collaborate with cross-functional partners to deliver customer value. You will design, build, and 
TOKENS: ['Our', 'team', 'is', 'looking', 'for', 'a', 'DevOps', 'Engineer', '.', 'You', 'will', 'collaborate', 'with', 'cross-functional', 'partners']
LABELS: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
--------------------------------------------------------------------------------
TEXT: We are hiring a Frontend Engineer to join a remote-first organization. You will communicate clearly with stakeholders and teammates. You will particip
TOKENS: ['We', 'are', 'hiring', 'a', 'Frontend', 'Engineer', 'to', 'join', 'a', 'remote-first', 'organization', '.', 'You', 'will', 'communicate']
LABELS: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
--------------------------------------------------------------------------------
TEXT: W

In [10]:
import os, json

os.chdir("/kaggle/working/jd-bias-detector")

for split in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    path = f"data/annotated/{split}"
    cleaned = []

    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                print(f"⚠️ Skipping bad JSON in {split}")
                continue

            tokens = obj.get("tokens")
            labels = obj.get("labels")

            if tokens is None or labels is None:
                print(f"⚠️ Missing keys in {split}")
                continue

            if len(tokens) != len(labels):
                raise ValueError(f"Mismatch in {split}: {len(tokens)} vs {len(labels)}")

            cleaned.append(obj)

    if len(cleaned) == 0:
        raise ValueError(f"{split} is empty after cleaning!")

    tmp_path = path + ".clean"
    with open(tmp_path, "w") as f:
        for obj in cleaned:
            f.write(json.dumps(obj) + "\n")

    os.replace(tmp_path, path)

    has_bias = sum(1 for s in cleaned if any(l != "O" for l in s["labels"]))
    print(f"✅ {split}: {len(cleaned)} samples ({has_bias} biased, {len(cleaned)-has_bias} neutral)")

✅ train.jsonl: 2614 samples (1307 biased, 1307 neutral)
✅ val.jsonl: 322 samples (161 biased, 161 neutral)
✅ test.jsonl: 338 samples (169 biased, 169 neutral)


In [11]:
import os, json, torch
import numpy as np
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from seqeval.metrics import f1_score, precision_score, recall_score
import torch.nn as nn

# ================= SETUP =================
os.environ["WANDB_DISABLED"] = "true"
os.chdir("/kaggle/working/jd-bias-detector")

# ================= CONFIG =================
CFG = {
    "model": "microsoft/deberta-v3-base",
    "output_dir": "/kaggle/working/jd-bias-detector/models/deberta-v3-final",
    "epochs": 5,
    "batch_size": 16,
    "lr": 2e-5,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_length": 256,
    "grad_accum": 2,
}

# ================= LABELS =================
LABELS = [
    "O","B-GENDER_CODED","I-GENDER_CODED",
    "B-AGEIST","I-AGEIST",
    "B-EXCLUSIONARY","I-EXCLUSIONARY",
    "B-ABILITY_CODED","I-ABILITY_CODED"
]

LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

# ================= DATA VALIDATION =================
def split_stats(path):
    total, biased = 0, 0
    with open(path) as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                total += 1
                if any(l != "O" for l in obj["labels"]):
                    biased += 1
    return total, biased

train_path = "data/annotated/train.jsonl"
val_path   = "data/annotated/val.jsonl"
test_path  = "data/annotated/test.jsonl"

t, b = split_stats(train_path)
print(f"Train: {t} (biased={b}, neutral={t-b})")

# ================= CLASS WEIGHTS (FIXED) =================
def compute_class_weights(path):
    counts = Counter()

    with open(path) as f:
        for line in f:
            if line.strip():
                counts.update(json.loads(line)["labels"])

    total = sum(counts.values())
    weights = torch.ones(len(LABELS))

    for lbl, idx in LABEL2ID.items():
        if lbl == "O":
            continue
        weights[idx] = min(5.0, (total / max(1, counts[lbl])) ** 0.5)

    return weights

# ================= DATASET =================
class BiasDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.samples = []

        with open(path) as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                if len(obj["tokens"]) != len(obj["labels"]):
                    continue
                self.samples.append(obj)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        enc = self.tokenizer(
            s["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=CFG["max_length"],
        )

        word_ids = enc.word_ids()
        labels = s["labels"]

        aligned = []
        prev = None

        for wid in word_ids:
            if wid is None or wid >= len(labels):
                aligned.append(-100)
            elif wid != prev:
                aligned.append(LABEL2ID.get(labels[wid], 0))
            else:
                aligned.append(-100)
            prev = wid

        return {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "labels": torch.tensor(aligned),
        }

# ================= METRICS =================
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    true_labels, true_preds = [], []

    for p_, l_ in zip(preds, labels):
        tl, tp = [], []
        for pi, li in zip(p_, l_):
            if li != -100:
                tl.append(ID2LABEL[li])
                tp.append(ID2LABEL[pi])
        if tl:
            true_labels.append(tl)
            true_preds.append(tp)

    return {
        "precision": precision_score(true_labels, true_preds, zero_division=0),
        "recall": recall_score(true_labels, true_preds, zero_division=0),
        "f1": f1_score(true_labels, true_preds, zero_division=0),
    }

# ================= TRAINER =================
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device),
            ignore_index=-100,
            label_smoothing=0.1  # 🔥 keep this
        )

        loss = loss_fn(
            logits.view(-1, logits.size(-1)),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# ================= MODEL =================
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(CFG["model"], model_max_length=256)

model = AutoModelForTokenClassification.from_pretrained(
    CFG["model"],
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# ================= DATA =================
train_ds = BiasDataset(train_path, tokenizer)
val_ds   = BiasDataset(val_path, tokenizer)

test_ds = None
if Path(test_path).exists():
    test_ds = BiasDataset(test_path, tokenizer)

# ================= TRAINING =================
args = TrainingArguments(
    output_dir=CFG["output_dir"],

    num_train_epochs=CFG["epochs"],
    per_device_train_batch_size=CFG["batch_size"],
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=CFG["grad_accum"],

    learning_rate=CFG["lr"],
    warmup_ratio=CFG["warmup_ratio"],
    weight_decay=CFG["weight_decay"],
    lr_scheduler_type="cosine",

    max_grad_norm=1.0,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),

    logging_steps=50,
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=compute_class_weights(train_path)
)

# ================= TRAIN =================
print("\n🚀 Training...")
trainer.train()

# ================= SAVE =================
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])

# ================= TEST =================
if test_ds:
    print("\n📊 Test Evaluation")
    results = trainer.evaluate(test_ds)
    for k, v in results.items():
        print(f"{k}: {round(v,4) if isinstance(v,float) else v}")

2026-03-18 14:26:42.681078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773844002.894534      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773844002.959532      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773844003.431267      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773844003.431318      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773844003.431321      55 computation_placer.cc:177] computation placer alr

Train: 2614 (biased=1307, neutral=1307)
Loading model...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

...

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)



🚀 Training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,1.807067,0.297674,0.297674,0.297674
2,2.261400,1.685570,0.945813,0.893023,0.918660
3,1.711300,1.674088,0.928251,0.962791,0.945205
4,1.675000,1.672015,0.963303,0.976744,0.969977
5,1.667000,1.671576,0.950673,0.986047,0.968037



📊 Test Evaluation


eval_loss: 1.6639
eval_precision: 0.9737
eval_recall: 0.9737
eval_f1: 0.9737
eval_runtime: 2.9213
eval_samples_per_second: 115.701
eval_steps_per_second: 2.054
epoch: 5.0


In [12]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model="/kaggle/working/jd-bias-detector/models/deberta-v3-final",
    aggregation_strategy="simple",
)

def clean_word(w):
    return w.replace("##", "").strip()

test_jds = [
    "We need someone who thrives in a demanding environment and loves intense challenges.",
    "Looking for a young, hungry developer who can crush it at a fast-paced startup.",
    "The ideal candidate is a collaborative team player who communicates clearly and supports their colleagues.",
    "We want a rockstar engineer with ninja-level skills who eats, sleeps and breathes code.",
    "Join our diverse team as a senior engineer. You will mentor others and drive technical decisions.",

    # 🔥 Generalization tests
    "We need someone who can adapt quickly in a dynamic work setting.",
    "Looking for candidates early in their career with strong motivation.",
    "We want someone who can handle a high workload and changing priorities.",
]

for jd in test_jds:
    results = ner(jd)

    print(f"\nJD: {jd}")

    if results:
        for r in results:
            word = clean_word(r["word"])
            print(f"  [{r['entity_group']}] '{word}' ({r['score']:.2f})")
    else:
        print("  No bias detected")


JD: We need someone who thrives in a demanding environment and loves intense challenges.
  [GENDER_CODED] 'demanding' (0.53)
  [ABILITY_CODED] 'intense challenges' (0.56)

JD: Looking for a young, hungry developer who can crush it at a fast-paced startup.
  [AGEIST] 'young' (0.70)
  [AGEIST] 'hungry' (0.67)
  [GENDER_CODED] 'crush it' (0.89)
  [ABILITY_CODED] 'paced' (0.32)

JD: The ideal candidate is a collaborative team player who communicates clearly and supports their colleagues.
  No bias detected

JD: We want a rockstar engineer with ninja-level skills who eats, sleeps and breathes code.
  [EXCLUSIONARY] 'rockstar' (0.68)
  [EXCLUSIONARY] 'ninja' (0.85)
  [EXCLUSIONARY] 'level' (0.81)

JD: Join our diverse team as a senior engineer. You will mentor others and drive technical decisions.
  No bias detected

JD: We need someone who can adapt quickly in a dynamic work setting.
  [GENDER_CODED] 'quickly' (0.39)
  [ABILITY_CODED] 'dynamic work' (0.51)

JD: Looking for candidates early

In [15]:
import os
import shutil

SRC = "models/deberta-v3-final"
DEST = "models/deberta-jd-bias-v1"

os.makedirs(DEST, exist_ok=True)

IMPORTANT_FILES = [
    "config.json",
    "pytorch_model.bin",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json"
]

for f in IMPORTANT_FILES:
    src_path = os.path.join(SRC, f)
    if os.path.exists(src_path):
        shutil.copy(src_path, DEST)

print("✅ Lightweight model saved:")
print(os.listdir(DEST))

✅ Lightweight model saved:
['spm.model', 'checkpoint-82', 'checkpoint-205', 'model.safetensors', 'checkpoint-123', 'checkpoint-41', 'training_args.bin', 'config.json', 'tokenizer_config.json', 'added_tokens.json', 'tokenizer.json', 'special_tokens_map.json']


In [14]:
import shutil
import os

base = "models/deberta-v3-final"

for f in os.listdir(base):
    if f.startswith("checkpoint"):
        shutil.rmtree(os.path.join(base, f))

print("🧹 Checkpoints removed")

🧹 Checkpoints removed


In [16]:
import shutil

shutil.make_archive(
    "deberta-jd-bias-v1",
    "zip",
    root_dir="models/deberta-jd-bias-v1"
)

print("✅ Ready for download")

✅ Ready for download


In [17]:
import shutil, os

base = "models/deberta-jd-bias-v1"

for f in os.listdir(base):
    if f.startswith("checkpoint") or f in ["training_args.bin"]:
        path = os.path.join(base, f)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

print("🧹 Cleaned")

🧹 Cleaned
